## USA Pay by Role

BLS OES May 2023 median annual wages. Entry = 10th pct, mid = 50th pct (median), senior = 90th pct.
Roles ordered from highest to lowest mid-career pay.

In [ ]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROLE_LABELS = {
    'software_engineer': 'Software Engineer', 'lawyer': 'Lawyer',
    'physician': 'Physician', 'financial_analyst': 'Financial Analyst',
    'registered_nurse': 'Registered Nurse', 'civil_engineer': 'Civil Engineer',
    'construction_laborer': 'Construction Laborer', 'farm_worker': 'Farm Worker',
    'manufacturing_worker': 'Manufacturing Worker', 'retail_worker': 'Retail Worker',
}

df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_usa_data.csv')
df['role_label'] = df['role'].map(ROLE_LABELS)

In [ ]:
mid_order = (
    df[df['career_stage'] == 'mid']
    .sort_values('median_salary_local', ascending=False)['role_label']
    .tolist()
)

stage_colors = {'entry': '#93c6ed', 'mid': '#2171b5', 'senior': '#08306b'}

fig = px.bar(
    df,
    x='role_label', y='median_salary_local', color='career_stage',
    barmode='group',
    category_orders={'role_label': mid_order, 'career_stage': ['entry', 'mid', 'senior']},
    color_discrete_map=stage_colors,
    title='USA Annual Salary by Role and Career Stage (BLS OES 2023)',
    labels={'role_label': 'Role', 'median_salary_local': 'Annual Salary (USD)', 'career_stage': 'Stage'},
)
fig.update_layout(xaxis_tickangle=-35)
fig.show()

In [ ]:
mid_df = df[df['career_stage'] == 'mid'].copy()
swe_mid = mid_df[mid_df['role'] == 'software_engineer']['median_salary_local'].values[0]
mid_df['swe_multiple'] = (swe_mid / mid_df['median_salary_local']).round(2)
mid_df = mid_df[mid_df['role'] != 'software_engineer'].sort_values('swe_multiple', ascending=True)
mid_df['role_label'] = mid_df['role'].map(ROLE_LABELS)

fig2 = px.bar(
    mid_df, x='swe_multiple', y='role_label', orientation='h',
    color='swe_multiple', color_continuous_scale='Blues',
    title=f'USA: SWE Mid-Career Pay as Multiple of Peer Roles (BLS OES 2023)<br>'
          f'<sup>SWE median = ${swe_mid:,}</sup>',
    labels={'swe_multiple': 'SWE / Role Salary Multiple', 'role_label': 'Role'},
)
fig2.add_vline(x=1.0, line_dash='dot', line_color='red', annotation_text='Equal pay')
fig2.show()